# Day 6 — Grover's Algorithm: Watching a Quantum Search

**Theme:** How a quantum computer "finds" an answer by turning probability toward it

## Student Instructions
- **Only edit the code cells labeled `EDIT THIS CELL ONLY`.**
- Run the notebook from top to bottom.
- When a cell says *predict*, write your guess down **before** you run it.

### If a cell runs too long
Some cells let you choose how many bits, qubits, or Grover rounds to use. **Big choices can take minutes, or even hours.**
Each of those cells says how long to expect, and the helper functions print an estimate before anything slow starts.
- A cell is still running while you see `[*]` next to it (Jupyter) or a spinning circle (VS Code).
- To stop it, press the **Interrupt** button (■) at the top of the notebook.
- If the notebook stops responding, **restart** it (Kernel → Restart), then re-run the Setup cell and the cells above where you were.

## Plan (about 75 minutes)
1. The search problem *(10 min)*
2. Amplitudes: the numbers behind the probabilities *(10 min)*
3. Grover's two moves: the oracle and the diffuser *(15 min)*
4. Watch Grover's algorithm run *(15 min)*
5. Too much of a good thing *(10 min)*
6. Scaling up, and what a real measurement shows *(10 min)*
7. Reflection *(5 min)*

> The pictures in this notebook copy the style of 3Blue1Brown's video
> [*But what is quantum computing? (Grover's Algorithm)*](https://www.youtube.com/watch?v=RQWpF2Gb-gU). Watch it after class!

## 0. Setup
Run this cell first. If it prints "ready", you are good to go.

In [ ]:
# DO NOT EDIT
# If Qiskit is not installed, remove the # from the next line and run this cell again:
# %pip install qiskit qiskit-aer matplotlib

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

simulator = Aer.get_backend("aer_simulator")   # the same simulator as Day 5
try:
    from grover_viz import *
    print("Qiskit and the Grover helpers are ready!")
except ModuleNotFoundError as error:
    print("Could not find", error.name)
    print("Make sure grover_viz.py is in the same folder as this notebook.")
    print("(Google Colab: click the folder icon on the left and upload grover_viz.py.)")

## 1. The Search Problem

In Week 1 you broke the Caesar cipher by **brute force**: try a key, check if the message
makes sense, try the next key...

Lots of problems look like this. Checking an answer is easy, but there is no shortcut to
*finding* it. Think of a combination lock: you can test any combination in a second,
but you have to search for the right one.

A classical computer checks **one guess at a time**. Let's measure how many guesses that takes.

### Try these ideas
- Run the cell several times. Does it always take the same number of tries?
- Change `n_bits` to 5, then 10, then 20. How do the tries grow?

> **Runtime warning:** every extra bit **doubles** the number of answers. Worst case on a typical laptop:
>
> | `n_bits` | 20 | 25 | 30 | 35 | 40 |
> |---|---|---|---|---|---|
> | worst-case time | instant | ~3 seconds | ~1.5 minutes | ~1 hour | ~1 day |
>
> Please stay at **25 or below**. (That slowness is exactly the problem Grover's algorithm attacks!)

In [ ]:
# EDIT THIS CELL ONLY
import random

n_bits = 3                              # the answer is a number with this many bits (25 or less!)
secret = random.randrange(2 ** n_bits)  # the computer hides a secret answer

def check(guess):
    """Says True or False. It never tells you the answer."""
    return guess == secret

tries = 0
for guess in range(2 ** n_bits):        # classical search: one guess at a time
    tries = tries + 1
    if check(guess):
        print("Found", guess, "after", tries, "tries, out of", 2 ** n_bits, "possible answers")
        break

### Reflection
- For `n_bits = 3` there are 8 possible answers. What is the **most** tries you could ever need? The **average**?
- If you add one more bit, what happens to the number of possible answers?

## 2. Amplitudes: the Numbers Behind the Probabilities

Earlier this week you saw that a Hadamard (`H`) gate puts a qubit into superposition, and measuring
gives 0 or 1 with 50/50 odds.

Behind every probability is a number called an **amplitude**:

$$\text{probability} = \text{amplitude}^2$$

Probabilities are never negative. **Amplitudes can be negative**, and that is the trick behind Grover's algorithm.

A real quantum computer **never shows you the amplitudes**. You only get measurement results.
A simulator lets us peek. Qiskit's `Statevector` gives us the amplitude of every possible answer.

### Try these ideas
- Pick any 3-bit `key` you like. The picture colors it **yellow**.
- *Predict:* what is each amplitude? Each probability?
- Sections 2 and 3 work best with **2 to 5 qubits** (the picture stops at 8). The key needs one bit per qubit.

In [ ]:
# EDIT THIS CELL ONLY
n_qubits = 3         # 2 to 5 works best
key = "101"          # the answer we are searching for (one bit per qubit)

qc = QuantumCircuit(n_qubits)
qc.h(range(n_qubits))           # Hadamard on every qubit: all answers at once
state = Statevector(qc)         # ask the simulator for every amplitude

show_state(state, key, caption="Hadamard on every qubit", show_geometry=False)

### Reflection
- How many bars are there, and why?
- Every amplitude is the same. What is it? (Hint: the 8 probabilities add up to 1.)
- If we measured now, would the key be more likely than any other answer?

## 3. Grover's Two Moves

Grover's algorithm repeats two steps over and over. One pass through both steps is called a
**round** (you'll also see it called a **Grover iteration**).

### Move 1: the Oracle — "flip the sign of the key"
The **oracle** is the quantum version of the `check()` function from Section 1.
It runs on **every answer at once**. It can't print "True", so it marks the key by
**flipping the sign of its amplitude**.

Look at the circuit. The `X` gates turn the key into `111`. The big multi-controlled gate flips
the sign only when **every** control is 1, like a Week 1 `AND` gate. Then the `X` gates
undo themselves.

In [ ]:
# DO NOT EDIT
oracle = phase_oracle(n_qubits, key)
display(oracle.draw())

state_after_oracle = state.evolve(oracle)     # apply the oracle to our state
show_state(state_after_oracle, key, caption="After the oracle", show_geometry=False)

### Reflection
- What happened to the yellow amplitude?
- Look at the **probability** of the key. Did it change? Why not?
- The red dashed line is the **mean** (average) amplitude. Did the oracle move it up or down?

### Move 2: the Diffuser — "reflect about the mean"
The **diffuser** flips every amplitude to the other side of the red mean line.
A bar that sits a distance *d* above the mean ends up *d* below it, and the other way around:

$$\text{new amplitude} = 2 \times \text{mean} - \text{old amplitude}$$

### Math Moment (predict before you run!)
With 8 answers, every amplitude starts at $1/\sqrt{8} \approx 0.354$. After the oracle, the key is $-0.354$.
1. What is the new mean? (Add the 8 amplitudes and divide by 8.)
2. Use the formula to find the key's new amplitude, and square it to get its probability.
3. Do the same for one of the other answers.

In [ ]:
# DO NOT EDIT
diff = diffuser(n_qubits)
display(diff.draw())

state_after_round_1 = state_after_oracle.evolve(diff)
show_state(state_after_round_1, key, caption="After round 1 (oracle + diffuser)", show_geometry=False)

### Your Turn: Round 2
*Predict:* if we apply the oracle and the diffuser **one more time**, will the key's probability go up or down?

In [ ]:
# EDIT THIS CELL ONLY
state_after_round_2 = state_after_round_1.evolve(oracle)   # oracle
state_after_round_2 = state_after_round_2.evolve(diff)     # diffuser

show_state(state_after_round_2, key, caption="After round 2", show_geometry=False)

## 4. Watch Grover's Algorithm Run

Now let's animate every step, the way 3Blue1Brown does it.

**The right side** is the bar picture you already know: the oracle flips the yellow bar across
the zero line, and the diffuser flips every bar across the mean line.

**The left side** is the big idea from the video. The whole quantum state is **one arrow**:
- **Up** (yellow axis) means "the key".
- **Right** (white axis) means "all the other answers, equally".
- The arrow starts at $|b\rangle$ (every answer equal), just a small angle $\theta$ above "others".
- **Oracle** = flip the arrow over the white line.
- **Diffuser** = flip the arrow over the teal $|b\rangle$ line.
- Two flips in a row = a **turn of $2\theta$** toward the key.
- The arrow's **height** is the key's amplitude, so **height² = P(key)**.

The goal: turn the arrow until it points straight up.

Use **▶** to play, or the **step** buttons to move one frame at a time.

> **How long will it take?** Build time depends mostly on the number of **rounds**: each round adds about 5 seconds.
>
> | rounds | 2 | 5 | 10 | 20 | 30 | more than ~65 |
> |---|---|---|---|---|---|---|
> | build time | ~10 s | ~25 s | ~45 s | ~1.5 min | ~2.5 min | won't start (over 5 min) |
>
> `rounds = None` picks the best number for you: 1 round for 2 qubits, 2 for 3 qubits, 4 for 5 qubits, 8 for 7 qubits.
> The animation works for **2 to 7 qubits**. It prints an estimate before it starts. Slower laptops take longer.

In [ ]:
# EDIT THIS CELL ONLY
n_qubits = 3       # 2 to 7
key = "101"        # one bit per qubit
rounds = None      # None = let the computer pick the best number of rounds (about 5 s per round)

animate_grover(n_qubits, key, iterations=rounds)

### Reflection
- During the **oracle**, the mean line moves. During the **diffuser**, it stays still. Why?
- Each ghost arrow on the left is an old state. How many flips did it take to reach the top?
- Where on the left picture can you "see" P(key)?
- Try `n_qubits = 2` (use a 2-bit key). Why does it find the key in just **one** round?

## 5. Too Much of a Good Thing

In a classical search, checking more never hurts. Grover's algorithm is different.
The arrow keeps turning by $2\theta$ every round, and it doesn't stop at the top.

### Try these ideas
- *Predict:* what happens to P(key) after 5 rounds with 3 qubits?
- Then try `rounds = 3`, `4`, `6`.

> **Runtime:** Each round adds about 5 seconds to the build, so **stay at 10 rounds or fewer** here (about 45 seconds).
> The graph in the next cell is instant, and it shows what happens with many more rounds.

In [ ]:
# EDIT THIS CELL ONLY
n_qubits = 3
key = "101"
rounds = 5         # 10 or fewer (about 5 s per round)

animate_grover(n_qubits, key, iterations=rounds)

In [ ]:
# DO NOT EDIT
plot_success_vs_rounds(n_qubits, key)

### Reflection
- In the arrow picture, why does too many rounds make the answer *less* likely?
- The graph goes up and down like a wave. Why does it repeat?

## 6. Scaling Up

More qubits means more answers $N$. More answers means a smaller starting angle $\theta$
(roughly $1/\sqrt{N}$), so we need more rounds to reach the top. But only about
$\frac{\pi}{4}\sqrt{N}$ of them.

In [ ]:
# DO NOT EDIT
print(f"{'qubits':>6} {'possible answers':>20} {'classical (average)':>20} {'Grover rounds':>14}")
for n in [2, 3, 5, 10, 20, 30, 40]:
    N = 2 ** n
    print(f"{n:>6} {N:>20,} {N // 2:>20,} {best_iterations(n):>14,}")

### Try these ideas
- Animate a **5-qubit** search (32 answers). Make the key 5 bits long.
- Compare the starting angle $\theta$ with the 3-qubit animation.

> **Runtime:** 5 qubits uses 4 rounds, about **25 seconds** to build. 6 qubits (6 rounds) is about 35 seconds,
> and 7 qubits (8 rounds) about 45 seconds. 7 is the largest the animation will draw.

In [ ]:
# EDIT THIS CELL ONLY
n_qubits = 5
key = "10110"
rounds = None

animate_grover(n_qubits, key, iterations=rounds)

### What a real quantum computer actually shows you

Everything so far used the simulator to **peek** at amplitudes. A real quantum computer can't do that.
Measuring gives **one** answer, and the superposition is gone.
So we run the circuit many times (**shots**) and count the results.

### Try these ideas
- Start with `shots = 1`, then `10`, `100`, `1000`.
- Try `rounds = 0` (no Grover at all). What do the results look like?
- Try a much bigger search, like 10 or 15 qubits. (No animation here, so it stays fast.)

> **How long will it take?** Shots barely matter. The number of **qubits** is what counts:
>
> | `n_qubits` | up to 16 | 18 | 20 | 22 | 24 or more |
> |---|---|---|---|---|---|
> | run time (best rounds) | under 1 s | ~2 s | ~10 s | ~2 minutes | won't start (over 10 minutes) |
>
> The histogram has one bar per answer, so above about 8 qubits it gets crowded. The yellow key still stands out.

In [ ]:
# EDIT THIS CELL ONLY
n_qubits = 5       # up to 20 runs quickly
key = "10110"      # one bit per qubit
rounds = None
shots = 10

In [ ]:
# DO NOT EDIT
# Build the circuit: Hadamards, then (Oracle + Diffuser) for each round, then measure
qc = grover_circuit(n_qubits, key, iterations=rounds)
qc.draw(fold=-1)

In [ ]:
# DO NOT EDIT
# Run it, just like on Day 5. transpile() translates our Oracle and Diffuser boxes
# into the basic gates the simulator understands.
job = simulator.run(transpile(qc, simulator), shots=shots)
counts = job.result().get_counts()
counts

In [ ]:
# DO NOT EDIT
# Like plot_histogram, but shows every possible answer and colors the key yellow
plot_counts(counts, marked=key)

### Reflection
- With only **1 shot**, could you trust the answer? How could you double-check it **cheaply**? (Hint: Section 1.)
- Why does it make sense to call Grover's algorithm "amplitude amplification"?

## 7. Reflection Questions
Answer these questions in complete sentences:
1. What kind of problem does Grover's algorithm solve?
2. What does the **oracle** do to the amplitudes? Why doesn't that change any probabilities by itself?
3. What does the **diffuser** do?
4. Describe one Grover round using the **arrow picture**.
5. Why can too many rounds make the result worse?
6. A search has about 1 million ($2^{20}$) answers. About how many checks would a classical computer need on average? About how many Grover rounds?

### Coming up in the Day 8 project
We use Grover's algorithm to crack the Week 1 Caesar cipher. The hard part:
building an oracle that recognizes the right key **without being told the key**.